# CottonGuard AI — Clean Workflow

This notebook is a concise entry point for the modular project. Repeated directory listings, duplicated experiments, temporary image inspections, and obsolete non-grouped splits have been removed from the main workflow.

**Important:** this notebook does not retrain the model automatically. Training and split regeneration are explicit commands because they can overwrite saved artifacts.

Class order for the existing checkpoint: `bacterial_blight`, `curl_virus`, `fussarium_wilt`, `healthy`.


## 1. Configure the project path

Set `PROJECT_ROOT` to the folder containing `app.py`, `src/`, `models/`, and `results/`.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()  # Change this if the notebook is opened from another directory.
print("Project root:", PROJECT_ROOT)
print("Root exists:", PROJECT_ROOT.exists())
print("Checkpoint exists:", (PROJECT_ROOT / "models/deep_learning/resnet18_finetuned_best.pth").exists())


## 2. Review the group-aware dataset splits

Use these splits for leakage-resistant evaluation. The earlier `train.csv`, `validation.csv`, and `test.csv` were generated by a simpler random split and should not replace the group-aware splits in final reporting.

In [ ]:
from src.config import RESULTS_DATA_DIR

for split_name in ["train_grouped.csv", "validation_grouped.csv", "test_grouped.csv"]:
    path = RESULTS_DATA_DIR / split_name
    if path.exists():
        frame = pd.read_csv(path)
        print(f"{split_name}: {len(frame)} images")
        print(frame["label"].value_counts().sort_index().to_dict())
    else:
        print(f"Missing: {path}")


## 3. Verify the trained ResNet18 checkpoint

This loads the saved weights and runs one real image through the same preprocessing used in the evaluation. It does not train or modify the checkpoint.

In [ ]:
from PIL import Image
from src.config import CLASS_NAMES, RESULTS_DATA_DIR
from src.predict import get_device, load_model, predict_image

test_csv = RESULTS_DATA_DIR / "test_grouped.csv"
test_frame = pd.read_csv(test_csv)
sample = test_frame.iloc[0]
image_path = Path(sample["path"])

print("True label:", sample["label"])
print("Image exists:", image_path.exists())
print("Class order:", CLASS_NAMES)

if image_path.exists():
    device = get_device()
    model = load_model(device=device)
    with Image.open(image_path) as image:
        result = predict_image(image, model, device=device)
    print("Prediction:", result["predicted_class"])
    print("Confidence:", f"{result['confidence']:.2%}")
    print("Probabilities:", result["probabilities"])


## 4. Review the saved results

This section reads saved metrics rather than recalculating them. Run the evaluation modules if you intentionally want to recalculate.

In [ ]:
from src.config import METRICS_DIR
import json

for filename in ["resnet18_final_test_results.csv", "traditional_ml_validation_results.csv"]:
    path = METRICS_DIR / filename
    print("\\n", filename)
    if path.exists():
        display(pd.read_csv(path))
    else:
        print("Not found:", path)

external_metrics = METRICS_DIR / "mendeley_external_metrics.json"
if external_metrics.exists():
    print("\\nExternal evaluation:")
    print(json.dumps(json.loads(external_metrics.read_text(encoding="utf-8")), indent=2))


## 5. Optional commands

Run from the project root in a terminal:

- `streamlit run app.py` — start the image-upload app.
- `python -m src.evaluation.evaluate_resnet` — evaluate the saved ResNet18 on `test_grouped.csv`.
- `python -m src.train_traditional_ml` — retrain traditional ML baselines.
- `python -m src.evaluation.evaluate_traditional_ml` — evaluate saved grouped traditional ML models.
- `python -m src.evaluation.evaluate_external` — rerun the fixed external evaluation.
- `python -m src.evaluation.gradcam /path/to/image.jpg` — generate an optional Grad-CAM overlay.
- `python -m src.train_resnet` — retrain ResNet18. **This can overwrite checkpoints.**
- `python -m src.data.prepare_splits` — regenerate grouped splits. **This can overwrite split CSVs.**

## Interpretation reminder

The saved internal grouped test result is 99.56% accuracy (225/226 correct). The external Mendeley result is 88.00% accuracy with 41.58% macro F1 and failures on the external curl-virus and Fusarium-wilt examples. Report these evaluations separately; softmax confidence is not a guarantee of correctness.
